In [1]:
# ============================================
# NOTEBOOK 2: FEATURE ENGINEERING
# Extracting shot distance, angle, and all
# variables needed to train our xG model
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsbombpy import sb

print("Libraries loaded")

Libraries loaded


In [3]:
# Reload shots from all 35 matches
# Same code as Notebook 1 — we reload rather
# than save/load because StatsBomb data is free
# and fast to reload

matches = sb.matches(competition_id=11, season_id=90)

all_shots = []
for match_id in matches['match_id']:
    events = sb.events(match_id=match_id)
    shots = events[events['type'] == 'Shot'].copy()
    shots['match_id'] = match_id
    all_shots.append(shots)

shots_df = pd.concat(all_shots, ignore_index=True)
print(f"Shots loaded: {len(shots_df)}")

C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Dell\football-xg-model\venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access on

Shots loaded: 839


In [4]:
# ============================================
# STEP 1: PARSE COORDINATES
# StatsBomb stores location as a list [x, y]
# so we need to extract them as separate columns
# ============================================

shots_df['x'] = shots_df['location'].apply(lambda loc: loc[0])
shots_df['y'] = shots_df['location'].apply(lambda loc: loc[1])

print("Sample coordinates:")
print(shots_df[['x', 'y']].head())
print(f"\nx range: {shots_df['x'].min():.1f} to {shots_df['x'].max():.1f}")
print(f"y range: {shots_df['y'].min():.1f} to {shots_df['y'].max():.1f}")

Sample coordinates:
       x     y
0  108.6  28.0
1  103.6  51.0
2  104.3  33.9
3   97.9  44.3
4  118.3  42.1

x range: 40.0 to 119.3
y range: 12.9 to 76.7


In [5]:
# ============================================
# STEP 2: SHOT DISTANCE
# Using Pythagoras theorem to calculate
# straight-line distance from shot location
# to the centre of the goal
#
# StatsBomb pitch: 120 x 80 yards
# Goal centre: approximately (120, 40)
# ============================================

GOAL_X = 120
GOAL_Y = 40

shots_df['distance'] = np.sqrt(
    (shots_df['x'] - GOAL_X)**2 +
    (shots_df['y'] - GOAL_Y)**2
)

print("Distance calculated")
print(f"\nAverage distance — all shots: {shots_df['distance'].mean():.1f} yards")
print(f"Average distance — goals only: {shots_df[shots_df['shot_outcome']=='Goal']['distance'].mean():.1f} yards")
print(f"Average distance — non-goals:  {shots_df[shots_df['shot_outcome']!='Goal']['distance'].mean():.1f} yards")

Distance calculated

Average distance — all shots: 17.9 yards
Average distance — goals only: 12.9 yards
Average distance — non-goals:  18.7 yards


In [6]:
# ============================================
# STEP 3: SHOT ANGLE
# The angle between the shot location and
# the goal centre tells us how "open" the
# goal looks to the shooter
#
# Central shots (small angle) = wider goal
# Wide shots (large angle) = narrower goal
# ============================================

shots_df['angle'] = np.arctan2(
    np.abs(shots_df['y'] - GOAL_Y),
    np.abs(shots_df['x'] - GOAL_X)
) * (180 / np.pi)

print("Angle calculated")
print(f"\nAverage angle — all shots:  {shots_df['angle'].mean():.1f} degrees")
print(f"Average angle — goals only: {shots_df[shots_df['shot_outcome']=='Goal']['angle'].mean():.1f} degrees")
print(f"Average angle — non-goals:  {shots_df[shots_df['shot_outcome']!='Goal']['angle'].mean():.1f} degrees")

Angle calculated

Average angle — all shots:  28.7 degrees
Average angle — goals only: 26.7 degrees
Average angle — non-goals:  29.0 degrees


In [7]:
# ============================================
# STEP 4: CREATE ALL FEATURES
# ============================================

# Target variable: 1 = Goal, 0 = Not Goal
shots_df['is_goal'] = (shots_df['shot_outcome'] == 'Goal').astype(int)

# Is it a header?
shots_df['is_header'] = (shots_df['shot_body_part'] == 'Head').astype(int)

# Was the player under pressure?
shots_df['under_pressure'] = shots_df['under_pressure'].fillna(False).astype(int)

# Shot technique encoded
shots_df['technique'] = shots_df['shot_technique'].fillna('Normal')

# Is it from open play?
shots_df['is_open_play'] = (shots_df['play_pattern'] == 'Regular Play').astype(int)

print("All features created")
print(f"\nGoals: {shots_df['is_goal'].sum()}")
print(f"Headers: {shots_df['is_header'].sum()}")
print(f"Under pressure: {shots_df['under_pressure'].sum()}")
print(f"Open play shots: {shots_df['is_open_play'].sum()}")

# Header conversion rate vs foot
header_rate = shots_df[shots_df['is_header']==1]['is_goal'].mean() * 100
foot_rate = shots_df[shots_df['is_header']==0]['is_goal'].mean() * 100
print(f"\nHeader conversion rate: {header_rate:.1f}%")
print(f"Foot shot conversion rate: {foot_rate:.1f}%")

All features created

Goals: 111
Headers: 105
Under pressure: 152
Open play shots: 305

Header conversion rate: 10.5%
Foot shot conversion rate: 13.6%


In [8]:
# ============================================
# STEP 5: BUILD FEATURE MATRIX
# Select only the columns our model needs
# ============================================

feature_cols = [
    'distance',
    'angle', 
    'is_header',
    'under_pressure',
    'is_open_play'
]

# Build clean feature matrix
shots_features = shots_df[feature_cols + ['is_goal', 
                                           'shot_statsbomb_xg',
                                           'player',
                                           'team',
                                           'x', 'y']].copy()

# Remove any rows with missing values
shots_features = shots_features.dropna(subset=feature_cols)

print("Feature matrix built")
print(f"Shape: {shots_features.shape}")
print(f"\nFeature summary:")
print(shots_features[feature_cols].describe().round(2))

# Save to data folder
shots_features.to_csv('../data/shots_features.csv', index=False)
print("\nSaved to data/shots_features.csv")

Feature matrix built
Shape: (839, 11)

Feature summary:
       distance   angle  is_header  under_pressure  is_open_play
count    839.00  839.00     839.00          839.00        839.00
mean      17.95   28.73       0.13            0.18          0.36
std        8.07   19.29       0.33            0.39          0.48
min        2.40    0.00       0.00            0.00          0.00
25%       11.92   12.57       0.00            0.00          0.00
50%       17.62   27.33       0.00            0.00          0.00
75%       23.14   42.23       0.00            0.00          1.00
max       80.20   82.34       1.00            1.00          1.00

Saved to data/shots_features.csv
